In [1]:
from google import genai
from google.genai import types

from model import TEFTaskAResponse, TEFTaskBResponse, TEFJudgeResponse
from prompt_taskA import eval1_system_instruction_taskA, eval1_taskA_prompt
from prompt_taskA import eval2_system_instruction_taskA, eval2_taskA_prompt
from prompt_taskA import judge_system_instruction_taskA, judge_prompt_taskA

from prompt_taskB import eval1_system_instruction_taskB, eval1_taskB_prompt
from prompt_taskB import eval2_system_instruction_taskB, eval2_taskB_prompt
from prompt_taskB import judge_system_instruction_taskB, judge_prompt_taskB

import pandas as pd
import copy
import json
from dotenv import load_dotenv
from pydantic import BaseModel, ValidationError
import re

p:\Conda\envs\tefcanada\lib\site-packages\pydantic\_internal\_fields.py:184: UserWarning: Field name "name" shadows an attribute in parent "Operation"; 
  warnings.warn(
p:\Conda\envs\tefcanada\lib\site-packages\pydantic\_internal\_fields.py:184: UserWarning: Field name "metadata" shadows an attribute in parent "Operation"; 
  warnings.warn(
p:\Conda\envs\tefcanada\lib\site-packages\pydantic\_internal\_fields.py:184: UserWarning: Field name "done" shadows an attribute in parent "Operation"; 
  warnings.warn(
p:\Conda\envs\tefcanada\lib\site-packages\pydantic\_internal\_fields.py:184: UserWarning: Field name "error" shadows an attribute in parent "Operation"; 
  warnings.warn(


In [2]:
def fully_flatten_pydantic_schema(model: type[BaseModel], nested_title: str = "CategoryFeedback") -> dict:
    """
    Fully flatten Pydantic v2 JSON schema:
    - Replaces $ref with the definition from $defs
    - Removes $defs and allOf
    - Sets nested_title for nested objects
    """
    raw_schema = model.model_json_schema()

    # Grab definitions
    defs = raw_schema.get("$defs", {})

    def _resolve_refs(schema: dict) -> dict:
        schema = copy.deepcopy(schema)
        if isinstance(schema, dict):
            # Resolve $ref
            if "$ref" in schema:
                ref_path = schema.pop("$ref")
                # In Pydantic v2, refs are like "#/$defs/CategoryFeedback"
                ref_name = ref_path.split("/")[-1]
                ref_schema = defs.get(ref_name, {})
                schema.update(_resolve_refs(ref_schema))
            
            # Flatten allOf
            if "allOf" in schema:
                merged = {}
                for subschema in schema.pop("allOf"):
                    merged.update(_resolve_refs(subschema))
                schema.update(merged)

            # Recurse
            for k, v in schema.items():
                schema[k] = _resolve_refs(v)

            # Set nested title if looks like CategoryFeedback
            if schema.get("type") == "object" and "properties" in schema:
                if set(["rating","justification","original","correction","recommendation"]).issubset(schema["properties"].keys()):
                    schema["title"] = nested_title

        elif isinstance(schema, list):
            schema = [_resolve_refs(item) for item in schema]

        return schema

    flat_schema = _resolve_refs(raw_schema)
    flat_schema.pop("$defs", None)
    return flat_schema

output_schema_taskA = fully_flatten_pydantic_schema(TEFTaskAResponse)
output_schema_taskB = fully_flatten_pydantic_schema(TEFTaskBResponse)

def flatten_pydantic_schema(model: type[BaseModel]) -> dict:
    raw_schema = model.model_json_schema()
    defs = raw_schema.get("$defs", {})

    def resolve(schema):
        if isinstance(schema, dict):
            schema = copy.deepcopy(schema)

            # Resolve $ref
            if "$ref" in schema:
                ref_name = schema["$ref"].split("/")[-1]
                return resolve(defs.get(ref_name, {}))

            # Merge allOf
            if "allOf" in schema:
                merged = {}
                for subschema in schema.pop("allOf"):
                    merged.update(resolve(subschema))
                schema.update(merged)

            # Convert prefixItems → items
            if "prefixItems" in schema:
                schema["items"] = schema.pop("prefixItems")

            # Recursively process dict
            for k, v in list(schema.items()):
                schema[k] = resolve(v)

            # Remove unwanted keys
            for key in ["$defs", "$ref", "allOf"]:
                schema.pop(key, None)

            return schema

        elif isinstance(schema, list):
            return [resolve(item) for item in schema]
        else:
            return schema

    flat = resolve(raw_schema)
    flat.pop("$defs", None)
    return flat

judge_output_schema = flatten_pydantic_schema(TEFJudgeResponse)

In [3]:
load_dotenv()

True

In [4]:
task_a_question = """

Type de document : le début d’un article de presse (rubrique faits-divers)
Objectif : écrire la suite de l’article (80 mots minimum)
Voici le début d’un article de presse.
Terminez cet article :
    en ajoutant à la suite un texte de 80 mots minimum ;
    en faisant plusieurs paragraphes.
Un chien perdu dans le parc
Hier après-midi, une femme a perdu son chien dans le parc Central. Elle a cherché partout mais ne l’a pas trouvé. Soudain, elle a entendu des aboiements…
"""

task_a_response = """

Malheureusement, le chien qui aboyait n’était pas le sien, mais un dalmatien qui se promenait avec son maître. Son maître portait des vêtements avec des motifs de dalmatiens et des chaussures violettes.

La femme a demandé au maître du dalmatien s’il avait vu son chien à elle : un grand caniche noir appelé Sarah. L’homme a répondu qu’il avait vu Sarah monter dans une voiture jaune.

La propriétaire du chien a immédiatement appelé le commissariat de police pour signaler l’enlèvementde son chien, enlevé par un kidnappeur dans une voiture jaune.

La police est toujours à la recherche du caniche noir Sarah et de son ravisseur.

Témoins ? Prière de contacter le commissariat de police du parc Central.
"""

word_count_taskA = len(task_a_response.split())

In [5]:
task_b_question = """

Type de document : une phrase extraite d’un journal

Objectif : exprimer son point de vue et le justifier (200 mots minimum)

Vous avez lu dans un journal l’affirmation suivante :

« Phrase issue d’un article de journal »

    → Écrivez une lettre au journal pour dire ce que vous en pensez (200 mots minimum).
    → Développez au moins 3 arguments pour défendre votre point de vue.

« Les examens scolaires ne mesurent pas vraiment l’intelligence. »
"""

task_b_response = """

En réponse à votre article au sujet de la pertinence des examens scolaires concernant l’évaluation de l’intelligence, j’aimerais partager mon opinion en tant que parent et enseignant.

D’abord, on pourrait s’interroger sur la définition de l’intelligence et de son caractère multiple. Si nous considérons que chaque étudiant est un individu unique, cela veut dire que tous les étudiants ont des talents et des styles d’apprentissage différents. C’est pourquoi les examens doivent offrir plusieurs façons de démontrer les savoirs acquis, pour une évaluation équitable qui tient compte des différences.

En effet, il y a plusieurs types d’examens pour évaluer les compétences : des examens à choix multiples, des examens de compréhension de texte, et des examens qui demandent une pensée rationnelle ou une pensée créative.

Je trouve que la compréhension de texte, ainsi que les questions à développement qui demandent aux élèves d’exprimer leurs idées, sont les examens les plus pertinents pour contribuer positivement au monde dans lequel nous vivons, peu importe le domaine d’apprentissage.

En ce sens, l’intelligence serait considérée comme une pensée claire, capable de trouver des solutions créatives face aux défis que nous connaissons. L’intelligence humaine doit être utile à la vie collective et non exister en huis clos. Elle serait donc liée à un nouvel humanisme, dédié au vivre-ensemble dans notre environnement.

Cela dit, les examens scolaires doivent poser des questions qui exigent une pensée claire et créative plutôt que factuelle et robotique, puisque l’intelligence humaine, en termes de conscience, est également toujours en développement.
"""

word_count_taskB = len(task_b_response.split())

In [6]:
taskA_content_eval1 = eval1_taskA_prompt.format(
    question=task_a_question,
    response=task_a_response,
    word_count=word_count_taskA
)

taskA_content_eval2 = eval2_taskA_prompt.format(
    question=task_a_question,
    response=task_a_response,
    word_count=word_count_taskA   
)

In [7]:
taskB_content_eval1 = eval1_taskB_prompt.format(
    question=task_b_question,
    response=task_b_response,
    word_count=word_count_taskB
)

taskB_content_eval2 = eval2_taskB_prompt.format(
    question=task_b_question,
    response=task_b_response,
    word_count=word_count_taskB
)

In [8]:
model_pro = 'gemini-2.5-pro'
model_flash = 'gemini-2.5-flash'

In [9]:
client = genai.Client()

response_eval1_taskA = client.models.generate_content(
    model=model_pro,
    config=types.GenerateContentConfig(
        system_instruction=eval1_system_instruction_taskA,
        response_mime_type='application/json',
        response_schema=output_schema_taskA),
    contents=taskA_content_eval1
)

response_eval2_taskA = client.models.generate_content(
    model=model_pro,
    config=types.GenerateContentConfig(
        system_instruction=eval2_system_instruction_taskA,
        response_mime_type='application/json',
        response_schema=output_schema_taskA),
    contents=taskA_content_eval2
)

In [10]:
response_eval1_taskB = client.models.generate_content(
    model=model_pro,
    config=types.GenerateContentConfig(
        system_instruction=eval1_system_instruction_taskB,
        response_mime_type='application/json',
        response_schema=output_schema_taskB),
    contents=taskB_content_eval1
)

response_eval2_taskB = client.models.generate_content(
    model=model_pro,
    config=types.GenerateContentConfig(
        system_instruction=eval2_system_instruction_taskB,
        response_mime_type='application/json',
        response_schema=output_schema_taskB),
    contents=taskB_content_eval2
)

In [23]:
def extract_feedback_df(response, metrics):
    """
    Extracts rating, justification, original, correction, and recommendation for each metric.
    Returns a DataFrame with the extracted info, including original and correction columns as lists.
    """
    match = re.search(r'\{.*\}', response.text, re.DOTALL)
    if not match:
        raise ValueError('No JSON object found in response.text')
    response_json_str = match.group(0)
    response_json = json.loads(response_json_str)
    results = {}
    for metric in metrics:
        data = response_json.get(metric, {})
        rating = data.get('rating')
        justification = data.get('justification')
        recommendation = data.get('recommendation')
        originals = data.get('original', [])
        corrections = data.get('correction', [])
        results[metric] = {
            'rating': rating,
            'justification': justification,
            'original': originals,
            'correction': corrections,
            'recommendation': recommendation
        }
    df = pd.DataFrame.from_dict(results, orient='index')
    df.index.name = 'metric'
    df.reset_index(inplace=True)
    return df


def extract_feedback_summary(df1, df2):
    """
    Extracts a summary of feedback from the DataFrames, flattening originals and corrections to single lists.
    """
    rating = float(round((df1.rating.mean()*0.5 + df2.rating.mean()*0.5),2))
    justification = ' '.join(df1.justification.tolist()) + ' '.join(df2.justification.tolist())
    recommendation = ' '.join(df1.recommendation.tolist()) + ' '.join(df2.recommendation.tolist())
    # Flatten all originals and corrections to single lists
    originals = [item for sublist in df1.original.tolist() + df2.original.tolist() if sublist for item in sublist]
    corrections = [item for sublist in df1.correction.tolist() + df2.correction.tolist() if sublist for item in sublist]
    return rating, justification, recommendation, originals, corrections

metrics_taskA = ['task_fulfillment', 'organization_coherence', 'content_relevance', 'vocabulary', 'grammar_syntax', 'cohesion', 'style_adaptability']
metrics_taskB = ['task_fulfillment', 'structure', 'argumentation', 'vocabulary', 'grammar_syntax', 'cohesion', 'tone', 'style_adaptability']

In [24]:
df_1_taskA = extract_feedback_df(response_eval1_taskA, metrics_taskA)
df_2_taskA = extract_feedback_df(response_eval2_taskA, metrics_taskA)
rating_taskA, justification_taskA, recommendations_taskA, originals_taskA, corrections_taskA = extract_feedback_summary(df_1_taskA, df_2_taskA)

df_1_taskB = extract_feedback_df(response_eval1_taskB, metrics_taskB)
df_2_taskB = extract_feedback_df(response_eval2_taskB, metrics_taskB)
rating_taskB, justification_taskB, recommendations_taskB, originals_taskB, corrections_taskB = extract_feedback_summary(df_1_taskB, df_2_taskB)

In [27]:
MIN_SCORE = 150
MAX_SCORE = 700
RANGE = MAX_SCORE - MIN_SCORE
P = 2.0 

def exponential_score(avg_rating, p=P):
    normalized = (avg_rating - 1) / 4 
    curved = normalized ** p
    final = MIN_SCORE + (curved * RANGE)
    return round(final)

In [28]:
final_raw_score = rating_taskA * 0.4 + rating_taskB * 0.6
print(f'Final raw score: {final_raw_score}')
final_score = exponential_score(final_raw_score)
print(f'Final score: {final_score}')

Final raw score: 4.556
Final score: 585


In [21]:
taskA_judge_content = judge_prompt_taskA.format(
    justification = justification_taskA,
    recommendations = recommendations_taskA,
    originals = originals_taskA,
    corrections = corrections_taskA)

taskB_judge_content = judge_prompt_taskB.format(
    justification = justification_taskB,
    recommendations = recommendations_taskB,
    originals = originals_taskB,
    corrections = corrections_taskB)

In [25]:
response_judge_taskA = client.models.generate_content(
    model=model_pro,
    config=types.GenerateContentConfig(
        system_instruction=judge_system_instruction_taskA,
        response_mime_type='application/json',
        response_schema=judge_output_schema),
    contents=taskA_judge_content
)

response_judge_taskB = client.models.generate_content(
    model=model_pro,
    config=types.GenerateContentConfig(
        system_instruction=judge_system_instruction_taskB,
        response_mime_type='application/json',
        response_schema=judge_output_schema),
    contents=taskB_judge_content
)

In [31]:
final_score_outof700 = round((rating_taskA*0.4 + rating_taskB*0.6)*6.99)
final_score_outof700

618

In [26]:
print(response_judge_taskA.text)

{
"justification": "The response effectively fulfills the task by logically continuing the story, exceeding the word count, and correctly adopting the 'fait-divers' format. The narrative is well-structured, with a creative and relevant plot development involving a potential dognapping. The writing style and tone are perfectly suited to the genre. However, the text contains a few weaknesses, including the use of an anglicism ('kidnappeur'), repetition of certain words ('chien', 'enlèvement'), and minor errors like a typo and redundancy ('son chien à elle'), which slightly diminish the overall quality.",
"recommendation": "To improve, proofread carefully to catch typos and spacing errors. Avoid anglicisms by using appropriate French terms like 'ravisseur'. Diversify your vocabulary by using synonyms to avoid repetition (e.g., 'l'animal' instead of 'le chien'). Work on making sentences more concise by eliminating redundancies such as 'à elle' and rephrasing to avoid repeating words like '

In [27]:
print(response_judge_taskB.text)

{
  "justification": "The text demonstrates a rich, sophisticated vocabulary and an excellent, semi-formal tone suitable for a letter to the editor. The writing style effectively mimics an authentic opinion piece. However, the response critically fails to meet key task requirements. It does not follow the required letter format, as it omits a salutation and closing. Furthermore, it does not present three distinct, well-structured arguments; instead, the ideas are merged into a single line of thought that is underdeveloped and lacks specific examples. While the grammar is generally strong, some awkward phrasing and sentence fragments detract from the overall quality. The use of transition words is inconsistent, failing to guide the reader through a logical progression of points.",
  "recommendation": "To improve, you must strictly adhere to the specified format, ensuring you include a salutation (e.g., 'Madame, Monsieur le Rédacteur en chef,') and a closing. Structure your response into